In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import nltk
import re
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords')
nltk.download('punkt')

print("✅ Libraries loaded!")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sujit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sujit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


✅ Libraries loaded!


In [2]:
# Dataset load பண்ணுங்கள்
df = pd.read_csv(r'C:\Users\sujit\OneDrive\Desktop\sentiment-analysis\data\Reviews.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (568454, 10)

Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [3]:
# Important columns மட்டும் எடுங்கள்
df = df[['Score', 'Text']].copy()

# Sentiment create பண்ணுங்கள்
# Score 4,5 → Positive, Score 1,2 → Negative, 3 → Remove
df = df[df['Score'] != 3]
df['Sentiment'] = df['Score'].apply(lambda x: 1 if x > 3 else 0)

# Sample எடுங்கள் (50,000 மட்டும் — faster training)
df = df.sample(50000, random_state=42)

print("✅ Data Ready!")
print(df['Sentiment'].value_counts())
df.head()

✅ Data Ready!
Sentiment
1    42147
0     7853
Name: count, dtype: int64


,Score,Text,Sentiment
443593,5,This is a very high quality dog food with meat...,1
136949,5,I love this cake mix and the other 3 mixes as ...,1
520459,4,A nice strong brew. I am new to Keurig and hav...,1
219515,5,I just found PB2 and PB2 with chocolate and I ...,1
471273,5,Delightful mint tea as one would expect. Note ...,1


In [4]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = text.lower()
    # Special characters remove
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Stopwords remove
    text = ' '.join([word for word in text.split() 
                     if word not in stop_words])
    return text

# Clean பண்ணுங்கள்
df['Clean_Text'] = df['Text'].apply(clean_text)

print("✅ Text Cleaned!")
df[['Text', 'Clean_Text']].head(3)

✅ Text Cleaned!


,Text,Clean_Text
443593,This is a very high quality dog food with meat...,high quality dog food meat fruits grains dog l...
136949,I love this cake mix and the other 3 mixes as ...,love cake mix mixes well incredible amazon off...
520459,A nice strong brew. I am new to Keurig and hav...,nice strong brew new keurig lived french press...


In [5]:
# Train/Test Split
X = df['Clean_Text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Model Train பண்ணுங்கள்
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Evaluate பண்ணுங்கள்
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"✅ Model Trained!")
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

✅ Model Trained!
Accuracy: 0.9197

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.56      0.68      1542
           1       0.92      0.99      0.95      8458

    accuracy                           0.92     10000
   macro avg       0.90      0.77      0.82     10000
weighted avg       0.92      0.92      0.91     10000

